<a href="https://colab.research.google.com/github/SANDULFERNANDO/Stellarx-Diagnostic-System/blob/feature%2FAugmentation/Augmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import shutil
import random

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
dataset_path = "/content/drive/MyDrive/dataset_tinea_project/Dataset_Collection_Tinea"

In [ ]:
base_dir = "/content/drive/MyDrive/dataset_tinea_project/Dataset_Collection_Tinea"

train_dir = os.path.join(base_dir, "train")
test_dir = os.path.join(base_dir, "test")

classes = ["Tinea","Leishmaniasis","Eczema_700"]

for folder in [train_dir,test_dir]:
    os.makedirs(folder,exist_ok=True)

    for c in classes:
        os.makedirs(os.path.join(folder,c),exist_ok=True)

In [ ]:
split_ratio = 0.8

random.seed(42)

for c in classes:

    source = os.path.join(dataset_path,c)

    images = os.listdir(source)

    random.shuffle(images)

    split = int(len(images)*split_ratio)

    train_images = images[:split]

    test_images = images[split:]

    for img in train_images:

        shutil.copy(
            os.path.join(source,img),
            os.path.join(train_dir,c,img)
        )

    for img in test_images:

        shutil.copy(
            os.path.join(source,img),
            os.path.join(test_dir,c,img)
        )

In [ ]:
for c in classes:

    print(c)

    print("Train :",len(os.listdir(os.path.join(train_dir,c))))

    print("Test :",len(os.listdir(os.path.join(test_dir,c))))

    print()

Tinea
Train : 148
Test : 37

Leishmaniasis
Train : 599
Test : 150

Eczema_700
Train : 560
Test : 140



In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import load_img,img_to_array

datagen = ImageDataGenerator(

    rotation_range=15,

    zoom_range=0.15,

    width_shift_range=0.1,

    height_shift_range=0.1,

    brightness_range=[0.8,1.2],

    horizontal_flip=True,

    fill_mode="nearest"
)

In [ ]:
tinea_train = os.path.join(train_dir,"Tinea")

In [ ]:
for image_name in os.listdir(tinea_train):

    image_path = os.path.join(tinea_train,image_name)

    image = load_img(image_path)

    image = img_to_array(image)

    image = np.expand_dims(image,axis=0)

    count = 0

    for batch in datagen.flow(

        image,

        batch_size=1,

        save_to_dir=tinea_train,

        save_prefix="aug",

        save_format="jpg"

    ):


        count += 1

        if count == 10:

            break

In [ ]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    validation_split=0.2
)

In [ ]:
train_generator = train_datagen.flow_from_directory(

    train_dir,

    target_size=(224,224),

    batch_size=32,

    class_mode="categorical",

    subset="training",

    shuffle=True
)

Found 2157 images belonging to 3 classes.


In [ ]:
validation_generator = train_datagen.flow_from_directory(

    train_dir,

    target_size=(224,224),

    batch_size=32,

    class_mode="categorical",

    subset="validation",

    shuffle=False
)

Found 538 images belonging to 3 classes.


In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(

    test_dir,

    target_size=(224,224),

    batch_size=32,

    class_mode="categorical",

    shuffle=False
)

Found 327 images belonging to 3 classes.
